# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ssaqlain09/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Lane:** Refresh / Content Opportunity Scoring, built on the warehouse release (not the 30k starter CSV).

**Contract answer 1 — what one row means.** The warehouse table's raw grain is **one content item on one
day for one client** (`report_date` + `client_hash_id` + `content_hash_id`). For modeling I aggregate that
to **one row = one content item (one `client_hash_id` + `content_hash_id`) over a fixed window**.

**Contract answer 2 — which table.** `fact_content_daily_performance` only (the daily search-performance
fact). I read just the columns I need, from one month partition.

**Contract answer 3 — which time window.** A mid-panel month, **`month=2026-03`**, split into two
non-overlapping 14-day windows: a **feature window (2026-03-01 → 2026-03-14)** and an **outcome window
(2026-03-15 → 2026-03-28)**. Both are two full weeks, so weekday patterns are equal on both sides. I do
*not* use the final month (June 2026): it is the sealed test month and the `_sample` table is that month.

The setup cell below connects DuckDB to the release. The token comes from a Colab Secret named `HF_TOKEN`
(or a hidden prompt) and is never written into this notebook.

In [2]:
%pip -q install duckdb huggingface_hub scikit-learn

import os, getpass

# Token order: env var -> Colab Secret -> hidden prompt. Never paste a token into a cell.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"
FACT_MONTH = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"

FEAT_START, FEAT_END = "2026-03-01", "2026-03-14"
OUT_START,  OUT_END  = "2026-03-15", "2026-03-28"

print("Connected. Slice = month", MONTH)
print("Feature window:", FEAT_START, "->", FEAT_END, "| Outcome window:", OUT_START, "->", OUT_END)

# Schema peek (metadata only, not one of the three verification queries)
con.sql(f"DESCRIBE SELECT * FROM {FACT_MONTH}").df()[["column_name", "column_type"]]

Paste your Hugging Face READ token (hf_...): ··········
Connected. Slice = month 2026-03
Feature window: 2026-03-01 -> 2026-03-14 | Outcome window: 2026-03-15 -> 2026-03-28


,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Contract answer 4 — what I rank / predict (label or proxy).** I rank content items by decline risk.
The label is a **future-window proxy measured inside the month**: `is_declining = 1` when a page's
impressions in the *outcome* window (Mar 15–28) are less than 80% of its impressions in the *feature*
window (Mar 1–14), for pages with at least 100 impressions in the feature window (a minimum-volume rule so
the ratio is not noise). It is an *observed* later outcome, not a bucket computed from the same window as
the features. It is still a proxy: a 14-day drop is not the same as a lasting decline.

**Contract answer 5 — one thing I deliberately exclude.** Everything from the outcome window is excluded
from features (the `imp_outcome` column and anything computed from it). It is the future: using it would
let the model read the answer. I also exclude the `_sample` table (it is the final month) and the two
pseudonymous ID columns as features.

| Bucket | Fields |
|---|---|
| **Feature** (knowable by Mar 14) | impressions, clicks, CTR, average position, active days — all from Mar 1–14 |
| **Label / proxy** | `is_declining` (from Mar 15–28 impressions vs Mar 1–14) |
| **Context** (group / split / read only) | `client_hash_id`, `content_hash_id` (pseudonyms — never features) |
| **Excluded** | `imp_outcome` and any ratio built from it (future), final-month `_sample` data (sealed test month) |

In [3]:
FIELDS = {
    "feature":  ["imp_feat", "clk_feat", "ctr_feat", "pos_feat", "active_days_feat"],
    "label":    ["is_declining"],
    "context":  ["client_hash_id", "content_hash_id"],
    "excluded": {"imp_outcome": "future window: it defines the label",
                 "_sample table": "it is the final month, kept as a sealed test month"},
}
for bucket, items in FIELDS.items():
    print(f"{bucket:9}", items)

feature   ['imp_feat', 'clk_feat', 'ctr_feat', 'pos_feat', 'active_days_feat']
label     ['is_declining']
context   ['client_hash_id', 'content_hash_id']
excluded  {'imp_outcome': 'future window: it defines the label', '_sample table': 'it is the final month, kept as a sealed test month'}


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three verification queries on `month=2026-03`. Each one checks one claim from the contract.
(Reading a month over the network can take a minute or two — that is normal.)

**Query 1 — grain.** Claim: one row really is one content item on one day for one client. If that is true,
grouping by those three columns and looking for counts above 1 returns **zero rows**.

In [4]:
q1 = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {FACT_MONTH}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print(f"Query 1 (grain): duplicate rows found = {len(q1)}  ->", "grain holds" if len(q1) == 0 else "GRAIN BROKEN, investigate")
q1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 1 (grain): duplicate rows found = 0  -> grain holds


,report_date,client_hash_id,content_hash_id,c


**Query 2 — slice size and date span.** Claim: the slice is March 2026 only. This gives the row count,
the first and last date, and how many clients and content items are in it.

In [5]:
q2 = con.sql(f"""
    SELECT COUNT(*)                        AS n_rows,
           MIN(report_date)                AS first_date,
           MAX(report_date)                AS last_date,
           COUNT(DISTINCT client_hash_id)  AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content_items
    FROM {FACT_MONTH}
""").df()
q2

,n_rows,first_date,last_date,n_clients,n_content_items
0,9841378,2026-03-01,2026-03-31,55,331437


**Query 3 — availability.** Claim: not every row has every measurement. Rows before a client's GA4 start
are filled with zeros and flagged `ga4_data_available = FALSE`, so a zero there is not "no engagement".
I filter with `IS TRUE` (which also handles NULL safely) and show how many rows survive.

In [6]:
q3 = con.sql(f"""
    SELECT COUNT(*)                                          AS total_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS rows_with_ga4,
           COUNT(*) FILTER (WHERE gsc_impressions > 0)        AS rows_with_gsc_impressions
    FROM {FACT_MONTH}
""").df()
q3["pct_ga4_survive"] = (100 * q3["rows_with_ga4"] / q3["total_rows"]).round(1)
q3["pct_gsc_survive"] = (100 * q3["rows_with_gsc_impressions"] / q3["total_rows"]).round(1)
q3

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,rows_with_ga4,rows_with_gsc_impressions,pct_ga4_survive,pct_gsc_survive
0,9841378,413966,3611061,4.2,36.7


### Five features (max), each knowable at the decision moment

The decision moment is **the end of 2026-03-14**: a reviewer looks at the pages and decides what to review
first. Every feature below uses only rows dated on or before that day.

| Feature | Knowable at the decision moment because… |
|---|---|
| `imp_feat` | it is the sum of impressions on Mar 1–14, all already recorded by Mar 14 |
| `clk_feat` | it is the sum of clicks on Mar 1–14, already recorded by Mar 14 |
| `ctr_feat` | it is `clk_feat / imp_feat`, both from the feature window only |
| `pos_feat` | it is the mean search position on Mar 1–14 (days with a real position), known by Mar 14 |
| `active_days_feat` | it counts days in Mar 1–14 with at least one impression, known by Mar 14 |

In [7]:
features = con.sql(f"""
    WITH agg AS (
        SELECT client_hash_id, content_hash_id,
               SUM(CASE WHEN report_date BETWEEN DATE '{FEAT_START}' AND DATE '{FEAT_END}'
                        THEN COALESCE(gsc_impressions, 0) ELSE 0 END)              AS imp_feat,
               SUM(CASE WHEN report_date BETWEEN DATE '{FEAT_START}' AND DATE '{FEAT_END}'
                        THEN COALESCE(gsc_clicks, 0) ELSE 0 END)                   AS clk_feat,
               AVG(CASE WHEN report_date BETWEEN DATE '{FEAT_START}' AND DATE '{FEAT_END}'
                         AND gsc_avg_position > 0 THEN gsc_avg_position END)       AS pos_feat,
               COUNT(DISTINCT CASE WHEN report_date BETWEEN DATE '{FEAT_START}' AND DATE '{FEAT_END}'
                         AND gsc_impressions > 0 THEN report_date END)             AS active_days_feat,
               SUM(CASE WHEN report_date BETWEEN DATE '{OUT_START}' AND DATE '{OUT_END}'
                        THEN COALESCE(gsc_impressions, 0) ELSE 0 END)              AS imp_outcome
        FROM {FACT_MONTH}
        WHERE report_date BETWEEN DATE '{FEAT_START}' AND DATE '{OUT_END}'
        GROUP BY 1, 2
    )
    SELECT *, clk_feat * 1.0 / NULLIF(imp_feat, 0) AS ctr_feat
    FROM agg
    WHERE imp_feat >= 100
""").df()

features["is_declining"] = (features["imp_outcome"] < 0.8 * features["imp_feat"]).astype(int)

FEATURES = ["imp_feat", "clk_feat", "ctr_feat", "pos_feat", "active_days_feat"]
before = len(features)
frame = features.dropna(subset=FEATURES).reset_index(drop=True)

print(f"Content items with >=100 impressions in the feature window: {before:,}")
print(f"After dropping rows with a missing feature:                 {len(frame):,}")
print(f"Base rate of is_declining:                                  {frame['is_declining'].mean():.3f}")
frame[["client_hash_id", "content_hash_id"] + FEATURES + ["is_declining"]].head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Content items with >=100 impressions in the feature window: 75,526
After dropping rows with a missing feature:                 75,526
Base rate of is_declining:                                  0.338


,client_hash_id,content_hash_id,imp_feat,clk_feat,ctr_feat,pos_feat,active_days_feat,is_declining
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,396.0,2.0,0.005051,4.342838,14,0
1,client_73cda7b4e4f265ea,content_05434271b257bb68,602.0,1.0,0.001661,5.070678,14,0
2,client_73cda7b4e4f265ea,content_d056587ff7faca0c,1145.0,9.0,0.007860,4.541584,14,0
3,client_73cda7b4e4f265ea,content_712c365258cee05c,2309.0,8.0,0.003465,4.899503,14,0
4,client_73cda7b4e4f265ea,content_3dba50ae010f3f30,226.0,1.0,0.004425,14.253117,14,1


### The trap: add one label-derived column on purpose

I add `leak_ratio = imp_outcome / imp_feat` as if it were a feature. The label is literally
`leak_ratio < 0.8`, so a model can read the answer off this column. I score a quick random forest with a
**client-grouped split** (whole clients held out) using ROC-AUC (0.5 = guessing, 1.0 = perfect), first with
the leak, then after deleting it. The honest number is the one that stays.

In [8]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

# Keep the quick experiment fast: sample up to 60,000 rows (split is still by whole clients).
work = frame.sample(n=min(len(frame), 60_000), random_state=42).reset_index(drop=True)

def quick_auc(df_, cols):
    y = df_["is_declining"]
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
    tr, te = next(splitter.split(df_, y, groups=df_["client_hash_id"]))
    model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1)
    model.fit(df_.iloc[tr][cols], y.iloc[tr])
    return roc_auc_score(y.iloc[te], model.predict_proba(df_.iloc[te][cols])[:, 1])

# 1) Deliberate leak
work["leak_ratio"] = work["imp_outcome"] / work["imp_feat"]
auc_leak = quick_auc(work, FEATURES + ["leak_ratio"])

# 2) Delete the leak, keep the honest number
work = work.drop(columns=["leak_ratio"])
auc_honest = quick_auc(work, FEATURES)

print(f"With the leaked column:    ROC-AUC = {auc_leak:.3f}   <- looks perfect, means nothing")
print(f"After deleting the leak:   ROC-AUC = {auc_honest:.3f}   <- the honest number")
print("Columns used now:", FEATURES)

With the leaked column:    ROC-AUC = 1.000   <- looks perfect, means nothing
After deleting the leak:   ROC-AUC = 0.602   <- the honest number
Columns used now: ['imp_feat', 'clk_feat', 'ctr_feat', 'pos_feat', 'active_days_feat']


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation of my slice: it is one short, mid-panel window, so my label is a short-horizon proxy.**
My outcome is a 14-day change inside a single month (March 2026), for clients that had data in March.
That means:

- **Short horizon and noise.** A 14-day drop can be a blip, not a lasting decline; I cannot tell them apart
  from one month.
- **Regression to the mean.** Pages with an unusually high feature window tend to drop afterwards by
  chance, so `imp_feat` may look predictive partly for that reason. I read the score as directional.
- **Unbalanced panel.** Clients have different history lengths and some have little or no data in this
  month, so the pages I see are not all clients equally.
- **No seasonality check.** One month cannot show whether March is unusual for a topic.

I will only claim observed, directional, decision-support results from this slice.

In [9]:
# Who is actually in my slice? (uses the frame already built — no extra warehouse query)
per_client = frame.groupby("client_hash_id").size()
print(f"Clients contributing rows: {per_client.shape[0]}")
print("Rows per client  min / median / max:",
      int(per_client.min()), "/", int(per_client.median()), "/", int(per_client.max()))
print(f"Share of rows from the single largest client: {per_client.max() / per_client.sum():.1%}")

Clients contributing rows: 37
Rows per client  min / median / max: 1 / 490 / 17280
Share of rows from the single largest client: 22.9%


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.